# LLM Performance Analysis

This notebook is for analyzing the student submission data to prepare for creating radar charts of LLM performance.

In [1]:
import json
import pandas as pd
import os

## 1. Load and Combine Data

First, we'll load the two main participation files and combine them into a single pandas DataFrame.

In [2]:
def load_data():
    """Loads and combines participation data from JSON files."""
    try:
        with open('../data/participation_a.json', 'r', encoding='utf-8') as f:
            data_a = json.load(f)
        with open('../data/participation_b.json', 'r', encoding='utf-8') as f:
            data_b = json.load(f)
        
        all_posts = data_a + data_b
        df = pd.DataFrame(all_posts)
        print(f"Successfully loaded and combined {len(df)} posts.")
        return df
    except FileNotFoundError as e:
        print(f"Error: {e}. Make sure the notebook is in a 'data_analysis' folder next to the 'data' folder.")
        return pd.DataFrame()

df = load_data()
df.head()

Successfully loaded and combined 326 posts.


,id,number,title,content,participation_type,author_name,author_id,llm_name,homework,links,categories,view_count,vote_count,reply_count,created_at,comments
0,6979789,81,Participation A (HW2): Grok,Q1. \n\nPart a of this problem was straight fo...,A,Leon Kornfeld,679296,Grok,HW2,"{'chat_links': [], 'drive_links': [], 'github_...","[errors, explanations, prompt engineering, cor...",139,0,1,2025-09-18T13:55:45.022993+10:00,"[{'author': 'Anant Sahai', 'content': 'We usua..."
1,7034106,102,Participation A Post,This is my participation A post. I have an exe...,A,Joe Berry,961809,Claude,HW1,"{'chat_links': [], 'drive_links': [], 'github_...","[errors, explanations, one-shot solving, promp...",195,0,0,2025-09-29T07:02:28.705216+10:00,[]
2,7049136,106,Special Participation A - Grok HW3,REFLECTION\n\n​​In completing the non-coding p...,A,Bruno Vieira,647432,Grok,HW3,"{'chat_links': [], 'drive_links': [], 'github_...","[hallucinations, explanations, iterative probl...",179,0,1,2025-10-01T09:44:26.995833+10:00,"[{'author': 'Anant Sahai', 'content': 'Great. ..."
3,7074543,116,Special Participation A: Deepseek with Deep Th...,Here is the online link: https://chat.deepseek...,A,Wesley Kai Zheng,647731,DeepSeek,HW0,{'chat_links': ['https://chat.deepseek.com/sha...,"[hallucinations, errors, explanations, one-sho...",232,0,1,2025-10-05T19:56:43.327041+11:00,"[{'author': 'Anant Sahai', 'content': 'Excelle..."
4,7077134,118,Participation A: HW 3 - Kimi,Executive Summary\n\nI tackled homework 3 with...,A,Deena Sun,983743,Kimi,HW3,"{'chat_links': [], 'drive_links': [], 'github_...","[hallucinations, errors, explanations, correct...",221,0,4,2025-10-06T08:50:54.954978+11:00,"[{'author': 'Deena Sun', 'content': 'Question ..."


## 2. Filter by LLM Name

This function allows us to filter the DataFrame to see posts related to a specific LLM.

In [3]:
def filter_by_llm(dataframe, llm_name):
    """Filters the DataFrame for a specific llm_name."""
    return dataframe[dataframe['llm_name'].str.lower() == llm_name.lower()]

## 3. Example: Show Posts for 'Gemini'

Let's test the function by filtering for all posts where the LLM was 'Gemini' and printing the title and content of each.

In [13]:
gemini_df = filter_by_llm(df, 'Mistral')

print(f"Found {len(gemini_df)} posts for the LLM 'Gemini'.\n")

for index, row in gemini_df.iterrows():
    print(f"--- POST TITLE: {row['title']} ---\n")
    print(f"This is a {"coding" if row['participation_type'] == 'B' else "math"} problem set\n")
    print(f"{row['content']}\n")
    print("="*50 + "\n")

Found 23 posts for the LLM 'Gemini'.

--- POST TITLE: Special Participation A: Mistral AI's Le Chat on HW3 ---

This is a math problem set

Here is the online link: https://chat.mistral.ai/chat/8c72d241-dd44-41a0-b8fc-a0469d84ff1d

Here is the annotated log:

Executive Summary:

From my observation, Le Chat was able to answer most written questions correctly on one shot. However, for questions that reference an external research paper, it could misunderstand the problem statement and draw something tangent to what the question is asking. In particular, it could reference a table on a different page or a formula in a different section. Doing some prompt engineering helps the model to reference the correct table/figure.

In addition, for questions that involve numerical counting, it could mistake the computation by a small margin, even after engineering the prompt. For instance, it could count something twice and mess up with the calculation. 


--- POST TITLE: Special Participation A: M

In [ ]:
import json

# Replace these with your actual file paths
file_paths = ['data/gemini_scores.json', 'data/chatgpt_scores.json', 'data/claude_scores.json']

# Initialize a dictionary to hold the sums and counts
sums = {}
counts = {}

# Iterate through each file
for path in file_paths:
    with open(path, 'r') as f:
        data = json.load(f)
        
        for llm, metrics in data.items():
            if llm not in sums:
                sums[llm] = {}
                counts[llm] = {}
            
            for metric, value in metrics.items():
                if metric not in sums[llm]:
                    sums[llm][metric] = 0.0
                    counts[llm][metric] = 0
                
                sums[llm][metric] += value
                counts[llm][metric] += 1

# Calculate averages and create the final dictionary
averaged_data = {}
for llm, metrics in sums.items():
    averaged_data[llm] = {}
    for metric, total_value in metrics.items():
        # Calculate average and round to nearest integer
        avg = total_value / counts[llm][metric]
        averaged_data[llm][metric] = int(round(avg))

# Output the result
print(json.dumps(averaged_data, indent=4))

# Optional: Save to a new file
# with open('averaged_results.json', 'w') as f:
#     json.dump(averaged_data, f, indent=4)